# பாடம் 13 - முகவர் நினைவகம்


## அமைப்பு

இந்த நோட்புக் **Microsoft Agent Framework** (MAF) பயன்படுத்தி **நிலைத்த நினைவகத்துடன்** ஒரு பயண முன்பதிவு முகவரியை எப்படி உருவாக்குவது என்பதை விளக்குகிறது.

செயற்கையாளரின் நினைவக வகைகள் — வேலை, குறுகிய காலம், மற்றும் நீண்ட காலம் — எப்படி ஒரு முகவர் தகவல்களை உரையாடல்கள் முழுவதும் வைத்திருக்கவும் பயன்படுத்தவும் பாதிக்கும் என்பதை நீங்கள் கற்றுக் கொள்வீர்கள்.

**முன் தேவைகள்:**
- நிறுவப்பட்ட உரையாடல் மாடல் கொண்ட Microsoft Foundry திட்டம் (உதா. `gpt-5-mini`).
- Azure CLI மூலம் உள்நுழைந்திருத்தல் — உங்கள் டெர்மினலில் `az login` ஐ இயக்கவும்.
- `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Microsoft Foundry திட்டத்தின் இறுதி புள்ளி.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — நீங்கள் நிறுவிய மாடலின் பெயர்.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ஏஜென்ட் நினைவக வகைகள்

AI ஏஜென்ட்கள் வெவ்வேறு வகையான நினைவகங்களை பயன்படுத்த முடியும், ஒவ்வொன்றும் தனித்துவமான ஒரு நோக்கத்தைக் கொண்டுள்ளது:

### பணிவிட நினைவகம்
உரையாடல் தண்டில் தான் — ஒரே அமர்வில் பரிமாறப்படும் செய்திகள். ஏஜென்ட் ஒரே தண்டிலில் முந்தைய செய்திகளுக்கு திரும்பிப் பார்த்து தொடக்கத்தை பராமரிக்க முடியும். MAF இல் இது **`agent.create_session()`** மூலம் உருவாக்கப்படுகிறது, இது ஒரு `AgentSession` ஐ நகலெடுக்கிறது.

### குறுகியகால நினைவகம்
ஒரு பணித்திட்டம் அல்லது அமர்வின் காலம் நிறைவு வரை நிலைக்கும் தகவல்கள், ஆனால் நிரந்தரமாக சேமிக்கப்படாது. உதாரணமாக, ஏஜென்ட் பல சுற்றுக்களுக்கிடையேயான திட்டமிடும் உரையாடலில் தகவல்களை சேகரித்து இறுதி பயணப்பட்டியலை உருவாக்க பயன்படுகிறது.

### நீண்டகால நினைவகம்
அமர்வுகளுக்கு __அடிக்கடி தொடர்புடைய__ விருப்பங்கள் மற்றும் உண்மைகள். திரும்பக் வரும் பயனர் தங்கள் உணவு தவறுகள் அல்லது பயண விதிகள் மீண்டும் கூற வேண்டியதில்லை. நீண்டகால நினைவு பொதுவாக வெளிப்புறக் கடையில் — தரவுத்தளம், கோப்பு, அல்லது வேக்டர் குறியீடு — வழியாக கிடைக்கிறது மற்றும் கருவிகளின் மூலம் ஏஜென்டிற்கு வழங்கப்படுகிறது.


## செயல்பாட்டு நினைவகத்துடன் அமர்வுகள்

நினைவகத்தின் எளிய வடிவம்தான் உரையாடல் அமர்வு. நீங்கள் ஒரே அமர்வை (`agent.create_session()` மூலம் உருவாக்கியதை) தொடர்ச்சியான `agent.run()` அழைப்புகளுக்கு அனுப்பும்போது, அந்த அமர்வின் முழு வரலாறையும் முகவர் காணலாம் மற்றும் பின் கூறிய விவரங்களை நினைவில் வைத்துக் கொள்ள முடியும்.

ஒரு பயண முகவரியை உருவாக்கி செயல்பாட்டு நினைவகத்தை விளக்குவோம்.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

முகவர் பட்ஜெட்டை சரியாக நினைவில் வைத்தார் ஏனெனில் இரு செய்திகளும் ஒரே அமர்வைக் பகிர்ந்துகொள்ளுகின்றன. இது **நடப்பு நினைவகம்** — இது அமர்வின் ஆயுளுக்கே மட்டுமே இருக்கும்.

### புதிய திருடன் என்ன நடக்கிறது?

நாம் ஒரு **புதிய** அமர்வை உருவாக்கினால், முகவருக்கு முந்தைய உரையாடலின் நினைவில்லை:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## நீண்டகால நினைவூட்டல் மாதிரியோடு

பயனர் விருப்பங்களை **அமைவுகளுக்கு மத்தியில்** நினைவில் வைக்க, உரையாடல் சரணை வெளியே இருக்கும் ஒரு நிரந்தர சேமிப்பகம் தேவை. செயற்கை நுண்ணறிவு இந்த சேமிப்பகத்தை அணுக **கருவிகள்** மூலம் — தகவலை சேமிக்க மற்றும் மீட்டெல்ல அழைக்கும் செயல்பாடுகளைப் பயன்படுத்திக் கொள்கிறது.

கீழே நாம் ஒரு எளிய நினைவகம் விருப்பக்கட்டளை சேமிப்பகத்தை உருவாக்கி (உற்பத்தி மண்டலத்தில் நீங்கள் இதை தரவுத்தளம் அல்லது வெக்டார் குறியீட்டுடன் ஆதரிக்க வேண்டும்) அதை செயற்கை நுண்ணறிவுக்கு பயன்படும் கருவிகளாக வெளிப்படுத்துகிறோம்.

### கட்டமைப்பு
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### காட்சிப்படம் 1 — முதன்முறையாக பயனர் நினைவுக் கொள்கையை பதிவு செய்கிறார்

சரா முதன்முறையாக வந்து சேர்ந்துள்ளார். முகவர்பவர் அவரது விருப்பங்களை கருவிகள் மூலம் சேமித்து, அவற்றைப் பயன்படுத்தி ஹோட்டல்களை பரிந்துரைக்க வேண்டும்.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### காட்சி 2 — சரா வாரங்களுக்கு பிறகு திரும்பி வந்தார்

சரா ஒரு **புது திரையில்** (புது அமர்வை உருவாக்குவதைக்) ஆரம்பிக்கிறார். வேலை நினைவகம் காலியாக உள்ளது, ஆனால் நீண்டகால விருப்பக் கடையில் அவளுடைய தகவல்கள் உள்ளன. முகவர் அதை மீண்டும் பெற்றுக் கொண்டு பரிந்துரைகளை தனிப்பயனாக்க பயன்படுத்த வேண்டும்.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் மூன்று வகையான முகவர் நினைவகங்களை மற்றும் அவற்றை Microsoft முகவர் கட்டமைப்புடன் எவ்வாறு செயல்படுத்துவது என்பதை கற்றுக்கொண்டீர்கள்:

| நினைவக வகை | MAF இயந்திரம் | ஆயுள்நாள் |
|---|---|---|
| **உருவாக்கும்** | `agent.create_session()` | ஒரே உரையாடல் |
| **குறுகிய காலம்** | ஒரு தொண்டை உள்ளேயான சேர்க்கப்பட்ட சூழல் | ஒரே பணி / அமர்வு |
| **நீண்டகாலம்** | `@tool` செயல்பாடுகள் மூலம் அணுகப்படும் வெளிப்புற சேமிப்பு | அமர்வுகளுக்கு இடையே |

### முக்கியமான எடுத்துக்காட்டுகள்
1. **`agent.create_session()`** உருவாக்கும் நினைவகத்தை வழங்குகிறது — முகவர் ஒரு அமர்வின் முழு உரையாடல் வரலாற்றையும் காண முடிகிறது.
2. **புதிய அமர்வுகள் சூழலை இழக்கின்றன** — நீண்டகால நினைவகமின்றி முகவர் முந்தைய உரையாடல்களை நினைவில் கொள்ள முடியாது.
3. **`@tool` செயல்பாடுகள்** இடைஞ்சலை நிரப்புகின்றன — அவை முகவருக்கு நிலையான சேமிப்பிலிருந்து தகவலை சேமிக்கவும் மீட்டெடுக்கவும் அனுமதிக்கின்றன.
4. **பிரத்தியேகப்படுத்தல் நேரத்துடன் மேம்படுகிறது** — அதிகமாக விருப்பங்கள் சேமிக்கப்பட்டால், முகவரின் பரிந்துரைகள் சிறப்பாக இருக்கும்.

### நிஜ உலக பயன்பாடுகள்
- **வாடிக்கையாளர் சேவை**: வாடிக்கையாளர் வரலாறு மற்றும் விருப்பங்களை நினைவில் வைக்கும்
- **தனிப்பட்ட உதவியாளர்கள்**: நாட்கள் அல்லது வாரங்களாக சூழல் நிலையை பராமரிக்கும்
- **சுகாதாரம்**: நோயாளி தகவல் மற்றும் விருப்பங்களை கண்காணிக்கும்
- **மின் வர்த்தகம்**: வரலாரின் அடிப்படையில் தனிப்பட்ட ஷாப்பிங்

### அடுத்த படிகள்
- நினைவக அகராதியை தரவுத்தளமோ வெக்டர் சேமிப்போடு மாற்றவும் (உதா: Azure AI Search)
- நேர- sensibilidad தகவல்களுக்கு நினைவக காலாவதியைச் சேர்க்கவும்
- பகிர்ந்த நினைவகத்துடன் பன்முகவர் அமைப்புகளை உருவாக்கவும்
- Cognee நோட்புக் மூலம் அறிவுத்தொகுப்பு ஆதாரமுடைய நினைவகத்தை ஆராயவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
